# 00 — ETL & Cleaning

Reads raw Steam JSON → flatten → clean types → feature engineering.

**Output** : `saveAsTable("steam_clean")`

Downstream notebooks consume via:
```
%run ./00_ETL
df = spark.table("steam_clean")
```

## Dataset provenance

Source: `steam_game_output.json` from SteamSpy via S3.

Based on `release_year` distribution (max = 2022, no 2023+ entries), this dataset was collected in **late 2022 or early 2023**. All temporal metrics (e.g., "years on market") must use ~2022 as reference year, not the current date.

In [0]:
from pyspark.sql import functions as F

In [0]:
filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df_raw = spark.read.format("json").option("inferSchema", "true").load(filepath)

df_flat = df_raw.select(F.col("id"), F.col("data.*"))
print(f"Total records: {df_flat.count()}")

Total records: 55691


In [0]:
df_games = df_flat.filter(F.col("type") == "game")
print(f"Games retained: {df_games.count()}")

Games retained: 55690


In [0]:
df_clean = (df_games
    .withColumn("price_usd",        F.col("price").cast("double") / 100)
    .withColumn("initialprice_usd", F.col("initialprice").cast("double") / 100)
    .withColumn("discount",         F.col("discount").cast("int"))
)

df_clean = (
    df_clean
    .withColumn(
        "release_year",
        F.expr("try_cast(get(split(release_date, '/'), 0) as int)")
    )
    .withColumn(
        "release_month",
        F.expr("try_cast(get(split(release_date, '/'), 1) as int)")
    )
)

owners_split = 'split(owners_no_comma, " [.][.] ")'
df_clean = (
    df_clean
    .withColumn(
        "owners_no_comma",
        F.regexp_replace(F.col("owners"), ",", "")
    )
    .withColumn(
        "min_owners",
        F.expr(f"try_cast(get({owners_split}, 0) as long)")
    )
    .withColumn(
        "max_owners",
        F.expr(f"try_cast(get({owners_split}, 1) as long)")
    )
    .withColumn(
        "avg_owners",
        ((F.col("min_owners") + F.col("max_owners")) / 2).cast("long")
    )
)

# Raw values mix formats ("21+", "MA 15+", "180"); normalize to integer age
age_str_col = F.regexp_replace(F.col("required_age"), "[^0-9]", "")
age_as_int = F.expr("try_cast(required_age_str as int)")
age_fixed = F.expr(
    'try_cast(regexp_replace(required_age_str, '
    '"^([^0]*)0", "$1") as int)'
)
df_clean = (df_clean
    .withColumn("required_age_str", age_str_col)
    .withColumn("required_age_clean",
        F.when(
            F.col("required_age_str") != '',
            F.when(age_as_int > 100, age_fixed)
            .otherwise(age_as_int)
        ).otherwise(None)
    )
)

df_clean = (df_clean
    .withColumn("total_reviews", F.col("positive") + F.col("negative"))
    .withColumn("rating_ratio",
        F.when(
            F.col("total_reviews") > 0,
            F.col("positive") / F.col("total_reviews")
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn("pos_neg_ratio",
        F.when(F.col("negative") > 0, F.col("positive") / F.col("negative"))
         .otherwise(None)
    )
)

df_clean = (df_clean
    .withColumn("platform_windows", F.col("platforms.windows"))
    .withColumn("platform_mac",     F.col("platforms.mac"))
    .withColumn("platform_linux",   F.col("platforms.linux"))
)

df_clean = (df_clean
    .withColumn("languages", F.split(F.col("languages"), ", "))
    .withColumn("genre", F.split(F.col("genre"), ", "))
)

df_clean = (df_clean
    .withColumn("is_free", (F.col("price_usd") == 0))
    .withColumn(
        "estimated_revenue",
        F.col("price_usd") * F.col("avg_owners")
    )
)

## Data quality: empty strings

Some text columns contain empty strings `""` instead of proper NULL values. These create phantom groups in `groupBy` operations and false elements in `explode`. We convert them to NULL so downstream notebooks can filter them explicitly.

In [0]:
print("Empty strings before cleanup:")
for col_name in ["name", "publisher", "developer"]:
    count = df_clean.filter(F.trim(F.col(col_name)) == "").count()
    print(f"  {col_name}: {count}")

print("\nEmpty elements in arrays before cleanup:")
for col_name in ["genre", "languages", "categories"]:
    count = df_clean.filter(F.array_contains(F.col(col_name), "")).count()
    print(f"  {col_name}: {count} rows with '' in array")

Empty strings before cleanup:
  name: 0
  publisher: 134
  developer: 126

Empty elements in arrays before cleanup:
  genre: 160 rows with '' in array
  languages: 10 rows with '' in array
  categories: 0 rows with '' in array


In [0]:
for col_name in ["name", "publisher", "developer"]:
    df_clean = df_clean.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.col(col_name))
    )

for col_name in ["genre", "languages", "categories"]:
    df_clean = df_clean.withColumn(
        col_name,
        F.filter(F.col(col_name), lambda x: x != "")
    )
    df_clean = df_clean.withColumn(
        col_name,
        F.when(F.size(F.col(col_name)) == 0, None).otherwise(F.col(col_name))
    )

## Column selection

The raw dataset contains columns not needed for analysis (e.g., `url`, `desc_snippet`, `reviews_url`). We keep only the columns relevant to the 12 study questions: pricing, temporal, audience, reviews, content, platform, and derived metrics.

In [0]:
df_clean = df_clean.select(
    "appid", "name", "publisher", "developer",
    "price_usd", "discount", "is_free",
    "release_year",
    "avg_owners",
    "positive", "negative", "total_reviews", "rating_ratio", "pos_neg_ratio",
    "genre", "categories", "languages",
    "required_age_clean", "platform_windows", "platform_mac", "platform_linux",
    "estimated_revenue",
    "ccu",
)

In [0]:
df_clean.printSchema()

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- discount: integer (nullable = true)
 |-- is_free: boolean (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- avg_owners: long (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- rating_ratio: double (nullable = true)
 |-- pos_neg_ratio: double (nullable = true)
 |-- genre: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- required_age_clean: integer (nullable = true)
 |-- platform_windows: boolean (nullable = true)
 |-- platform_mac: boolean (nullable = true)
 |-- platform_linux

In [0]:
display(df_clean.limit(5))

appid,name,publisher,developer,price_usd,discount,is_free,release_year,avg_owners,positive,negative,total_reviews,rating_ratio,pos_neg_ratio,genre,categories,languages,required_age_clean,platform_windows,platform_mac,platform_linux,estimated_revenue,ccu
10,Counter-Strike,Valve,Valve,9.99,0,false,2000,15000000,201215,5199,206414,0.9748127549487923,38.70263512213887,List(Action),"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)","List(English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean)",0,true,true,true,1.4985E8,13990
1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,9.99,0,false,2021,10000,27,5,32,0.84375,5.4,"List(Action, Adventure, Indie)","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)","List(English, Korean, Simplified Chinese)",0,true,false,false,99900.0,0
1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,5.99,70,false,2020,350000,4032,646,4678,0.8619067977768277,6.241486068111455,"List(Adventure, Indie, RPG, Strategy)","List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)","List(Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil)",0,true,false,false,2096500.0,99
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,19.99,0,false,2020,150000,1575,115,1690,0.9319526627218935,13.695652173913043,"List(Action, Indie, Simulation, Strategy)","List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",List(English),0,true,true,false,2998499.9999999995,76
1000040,细胞战争,DoubleC Games,DoubleC Games,1.99,0,false,2019,10000,0,1,1,0.0,0.0,"List(Action, Casual, Indie, Simulation)",List(Single-player),List(Simplified Chinese),0,true,false,false,19900.0,0


## Null profiling

Some NULLs are expected by design:
- `release_month` / `release_year`: unparseable date formats in source data
- `rating_ratio`: games with zero reviews (division by zero protection)
- `pos_neg_ratio`: games with zero negative reviews (division by zero protection)
- `publisher`, `developer`, `genre`, `languages`: originally empty strings, converted to NULL above

This profiling step quantifies missing data to inform downstream filtering decisions.

In [0]:
total = df_clean.count()
engineered_cols = [
    "price_usd",
    "release_year",
    "avg_owners",
    "required_age_clean",
    "rating_ratio", "total_reviews",
    "estimated_revenue",
    "platform_windows", "platform_mac", "platform_linux",
    "is_free",
]

null_row = df_clean.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in engineered_cols]
).first()

stats = [(c, null_row[c], round(null_row[c] / total * 100, 1)) for c in engineered_cols]
df_nulls_eng = spark.createDataFrame(stats, ["column", "null_count", "null_pct"])
display(df_nulls_eng)

column,null_count,null_pct
price_usd,0,0.0
release_year,99,0.2
avg_owners,0,0.0
required_age_clean,0,0.0
rating_ratio,163,0.3
total_reviews,0,0.0
estimated_revenue,0,0.0
platform_windows,0,0.0
platform_mac,0,0.0
platform_linux,0,0.0


In [0]:
all_cols = df_clean.columns

null_row = df_clean.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in all_cols]
).first()

stats = [(c, null_row[c], round(null_row[c] / total * 100, 1)) for c in all_cols]
df_nulls = spark.createDataFrame(stats, ["column", "null_count", "null_pct"])
display(df_nulls)

column,null_count,null_pct
appid,0,0.0
name,0,0.0
publisher,134,0.2
developer,126,0.2
price_usd,0,0.0
discount,0,0.0
is_free,0,0.0
release_year,99,0.2
avg_owners,0,0.0
positive,0,0.0


In [0]:
any_null = df_clean.filter(
    F.greatest(*[F.col(c).isNull().cast("int") for c in all_cols]) == 1
).count()
print(f"\nRows with at least one NULL: {any_null} / {total} ({round(any_null / total * 100, 1)}%)")

review_null = df_clean.filter(
    F.col("rating_ratio").isNull() | F.col("pos_neg_ratio").isNull()
).count()
print(f"Rows with NULL rating_ratio or pos_neg_ratio: {review_null} / {total} ({round(review_null / total * 100, 1)}%)")


Rows with at least one NULL: 9952 / 55690 (17.9%)
Rows with NULL rating_ratio or pos_neg_ratio: 8797 / 55690 (15.8%)


## Profiling observations

**Data quality:** 17.9% of rows (9,952) have at least one NULL. Most NULLs come from `pos_neg_ratio` (15.8%) which is expected by design — games with zero negative reviews have no meaningful ratio.

**Metric coverage:**
- `rating_ratio`: 99.7% coverage (163 games with zero reviews excluded) — highly reliable
- `pos_neg_ratio`: 84.2% coverage (8,797 games with zero negative reviews excluded) — downstream averages should note this coverage gap

**Cleaned columns:** `publisher` (134), `developer` (126), `genre` (160), `languages` (10) had empty strings converted to NULL. These represent <0.3% of the dataset each — negligible, but downstream `groupBy`/`explode` operations must filter NULLs to avoid phantom groups.

**`categories`:** 970 NULLs (1.7%) are native — not from blank cleanup. Games without categories will be excluded from category-based analysis.

**Distribution skew:** `avg_owners` has median=10,000 but mean=133,514 — the market is dominated by a few mega-hits. Median-based analysis is more representative than mean for this column.

In [0]:
display(
    df_clean.select(
        "price_usd", "release_year", "avg_owners",
        "required_age_clean", "total_reviews", "rating_ratio", "pos_neg_ratio",
    ).summary()
)

summary,price_usd,release_year,avg_owners,required_age_clean,total_reviews,rating_ratio,pos_neg_ratio
count,55690,55591,55690,55690,55690,55527,46893
mean,7.732988687371752,2018.6526955802199,133513.55719159634,0.18701741784880588,1712.608044532232,0.7365350137972013,5.826375958811493
stddev,10.931394858517274,2.7183214188140354,1849924.5941899023,1.7279623328623583,35687.922958808886,0.24232000601177253,10.522802907693231
min,0.0,1997,10000,0,0,0.0,0.0
25%,1.34,2017,10000,0,7,0.6086956521739131,1.3125
50%,4.99,2019,10000,0,26,0.7985436893203883,3.0
75%,9.99,2021,35000,0,145,0.9249448123620309,6.5
max,999.0,2022,350000000,35,6730438,1.0,937.0


In [0]:
spark.sql("DROP TABLE IF EXISTS steam_clean")
df_clean.write.mode("overwrite").saveAsTable("steam_clean")
print("Table 'steam_clean' — ready.")
print("Downstream notebooks: df = spark.table('steam_clean')")

Table 'steam_clean' — ready.
Downstream notebooks: df = spark.table('steam_clean')
